# Backoffice Ticket Priority Classification

### Operational Efficiency Backoffice Automation — Banking-AI

This notebook follows the Energy-AI philosophy: Beginner-friendly, synthetic data, EDA, modeling, and interpretation.

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of operational efficiency and back-office automation terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the NLP task in the context of operational efficiency and back-office automation and how text features are extracted.
- Implement a text classification pipeline and evaluate accuracy on representative banking queries.
- Evaluate robustness to varied phrasing and identify failure modes relevant to customer-facing deployment.

**Estimated time:** 35–45 minutes

**Collection context:** Invoice processing, process optimisation, ticket management, and resource allocation in banking operations.

## Step 1 — Banking Problem Introduction

Backoffice teams receive hundreds of requests. Not all are equally urgent. This notebook uses text classification to automatically assign a priority level to incoming tickets.

**Primary stakeholders:** operations managers, process engineers, back-office team leads, and IT automation specialists.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Dataset

Synthetic ticket data.

In [ ]:
tickets = [
    ('Payment failed for corporate client', 'High'),
    ('Urgent: System down in branch 5', 'High'),
    ('How to change my address', 'Low'),
    ('Request for new checkbook', 'Low'),
    ('Error in monthly statement', 'Medium'),
    ('Duplicate transaction reported', 'Medium'),
    ('Immediate action: Fraud suspected', 'High'),
    ('Query about interest rates', 'Low'),
    ('Cannot login to portal', 'Medium'),
    ('Account closure request', 'Medium')
] * 20
df = pd.DataFrame(tickets, columns=['text', 'priority'])

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Look at the distribution of priorities.

In [ ]:
print(df['priority'].value_counts())

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform(df['text'])
y = df['priority']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the most frequent class ---
from collections import Counter
target_col = df.columns[-1]
majority_class, majority_n = Counter(df[target_col]).most_common(1)[0]
print(f"Majority-class baseline: always predict '{majority_class}' -> accuracy {majority_n/len(df):.3f}")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# Test on new data
new_ticket = ['Urgent: Credit card stolen']
print(f'Prediction for "{new_ticket[0]}":', model.predict(tfidf.transform(new_ticket))[0])

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

The evaluation metrics and visualisations above show the model's behaviour. In production, SHAP values or partial dependence plots would provide deeper per-prediction explanations.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: classify new queries ---
print("Operational demonstration — real-time intent classification:\n")
sample_queries = [
    "Can you show me my account balance?",
    "I need to transfer money to savings",
    "My card was stolen, please help",
    "What interest rates do you offer?",
    "I want to close my account",
]
for q in sample_queries:
    intent = model.predict([q])[0]
    print(f'  "{q}"  ->  {intent}')

print("\nIn production, predicted intents would route queries to the correct service channel.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end operational efficiency and back-office automation workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Automation of back-office tasks may displace workers and requires thoughtful transition planning.
- Errors in automated data extraction can cascade through downstream financial processes.
- Algorithmic resource allocation must be fair and transparent to affected teams and branches.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in operational efficiency and back-office automation?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.